# Reset & Fresh Clone Notebook (`unclone_clone.ipynb`)

**Repository:** [SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages](https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git)

This notebook completely wipes old/nested repository folders, frees system memory and disk cache, and performs a fresh download/clone of the latest code and datasets from GitHub.

In [ ]:
# Step 1 — Unclone: Free Memory & Wipe Old Directories
import os, sys, shutil, gc
from pathlib import Path

# 1. Reset kernel working directory to system home root
try:
    home_dir = Path.home()
    if (Path('/home/jovyan')).exists():
        home_dir = Path('/home/jovyan')
    elif (Path('/content')).exists():
        home_dir = Path('/content')
    os.chdir(home_dir)
except Exception as e:
    print(f'[WARN] Directory reset fallback: {e}')

print(f'[INFO] Working directory reset to: {Path.cwd()}')

# 2. Force delete old repository directories and cached archives
repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'
dirs_to_clean = [repo_name, f'{repo_name}-main', 'repo.zip']

for item in dirs_to_clean:
    if os.path.isdir(item):
        print(f'Removing directory: {item} ...')
        shutil.rmtree(item, ignore_errors=True)
    elif os.path.isfile(item):
        print(f'Removing file: {item} ...')
        os.remove(item)

# 3. Garbage collect & free GPU memory cache if active
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('[INFO] GPU memory cache cleared successfully.')
except Exception:
    pass

print('🧹 Workspace reset & memory cleanup complete!')


In [ ]:
# Step 2 — Clone: Fresh Repository Download & Extraction
import urllib.request, zipfile

repo_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git'
zip_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages/archive/refs/heads/main.zip'
repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'

# Try git clone first, fallback to pure Python zip download if git command is missing
print('Cloning repository from GitHub...')
clone_status = os.system(f'git clone {repo_url}')

if clone_status != 0 or not os.path.exists(repo_name):
    print('[INFO] git command unavailable or failed. Downloading repository archive directly...')
    urllib.request.urlretrieve(zip_url, 'repo.zip')
    with zipfile.ZipFile('repo.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    if os.path.exists(f'{repo_name}-main'):
        os.rename(f'{repo_name}-main', repo_name)

# Change directory to freshly cloned project root
%cd {repo_name}
print('\n✅ Fresh Clone Complete! Current project root:')
!pwd


In [ ]:
# Universal Self-Healing Path Setup (Prevents FileNotFoundError on deleted CWD)
import os, sys
from pathlib import Path

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir('/home/jovyan')
    _cwd = Path.cwd()

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'

if (_cwd / 'src').exists():
    BASE_DIR = _cwd
elif (_cwd.parent / 'src').exists():
    BASE_DIR = _cwd.parent
elif (_cwd / repo_name / 'src').exists():
    BASE_DIR = _cwd / repo_name
elif (Path('/home/jovyan') / repo_name / 'src').exists():
    BASE_DIR = Path('/home/jovyan') / repo_name
else:
    BASE_DIR = Path('/home/jovyan')

os.chdir(BASE_DIR)

for path_to_add in [str(BASE_DIR), str(BASE_DIR / 'src')]:
    if path_to_add not in sys.path:
        sys.path.insert(0, path_to_add)

DATA_DIR        = BASE_DIR / 'data' / 'raw'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
CHECKPOINTS_DIR = BASE_DIR / 'models' / 'checkpoints'
SRC_PATH        = BASE_DIR / 'src' / 'nllb_pipeline.py'

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Training set.csv'
VAL_PATH   = DATA_DIR / 'Validation set.csv'
TEST_PATH  = DATA_DIR / 'Test set.csv'

print(f'BASE_DIR : {BASE_DIR.resolve()}')
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, SRC_PATH]:
    status = 'OK' if p.exists() else 'MISSING'
    rel_p = p.relative_to(BASE_DIR) if p.is_relative_to(BASE_DIR) else p
    print(f'  [{status}] {rel_p}')
